In [1]:
import torch
import torch.nn as nn

# 4 layer with [size_of_each_layer] size
class FeedforwardWithSkipConn(nn.Module):
    def __init__(self, size_of_each_layer, use_shortcut=False):
        super().__init__()
        self.use_shortcut = use_shortcut
        self.layers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(size_of_each_layer[0], size_of_each_layer[1]),
                nn.GELU(),
            ),
            nn.Sequential(
                nn.Linear(size_of_each_layer[1], size_of_each_layer[2]),
                nn.GELU(),
            ),
            nn.Sequential(
                nn.Linear(size_of_each_layer[2], size_of_each_layer[3]),
                nn.GELU(),
            ),
            nn.Sequential(
                nn.Linear(size_of_each_layer[3], size_of_each_layer[4]),
                nn.GELU(),
            ),
            nn.Sequential(
                nn.Linear(size_of_each_layer[4], size_of_each_layer[5]),
                nn.GELU(),
            ),
        ])
        
    def forward(self, x):
        for layer in self.layers:
            layer_output = layer(x)
            
            if self.use_shortcut and x.shape == layer_output.shape:   # after finishing feedforward layer
                x = x + layer_output
            else:
                x = layer_output
                
        return x

In [3]:
size_of_each_layer = [3,3,3,3,3,1] # 4 layers
sample_input = torch.tensor([1.0, 1.0, -1.0])

In [5]:
def print_gradients(model, x):
    output = model(x)
    target = torch.tensor([0.0])

    loss = nn.MSELoss()
    loss = loss(output, target)
    
    loss.backward()
    
    for name, param in model.named_parameters():
        if 'weight' in name:
            print(f"{name} has gradient mean of {param.grad.abs().mean().item()}")

# Without Skip Connection

In [6]:
feedForwardWithoutSkip = FeedforwardWithSkipConn(size_of_each_layer, use_shortcut=False)
print_gradients(feedForwardWithoutSkip, sample_input)

layers.0.0.weight has gradient mean of 0.0004419709730427712
layers.1.0.weight has gradient mean of 0.0003148552495986223
layers.2.0.weight has gradient mean of 0.00045698892790824175
layers.3.0.weight has gradient mean of 0.0015616791788488626
layers.4.0.weight has gradient mean of 0.011077155359089375


# Using skip Connection (reduce vanishing gradient problems)

In [7]:
feedForwardWithSkip = FeedforwardWithSkipConn(size_of_each_layer, use_shortcut=True)
print_gradients(feedForwardWithSkip, sample_input)

layers.0.0.weight has gradient mean of 0.045053694397211075
layers.1.0.weight has gradient mean of 0.08253848552703857
layers.2.0.weight has gradient mean of 0.041262805461883545
layers.3.0.weight has gradient mean of 0.6341586709022522
layers.4.0.weight has gradient mean of 2.0364668369293213
